# Lesson 10: Agent State, Memory & Persistence

## Lab 10: Build a Persistent Agent

Real agents work on tasks that take time: many steps, many tool calls, sometimes waiting hours for a human. Things go wrong in the middle. A server restarts, a laptop closes, an API call crashes. Without persistence, all the work is lost and the agent starts again from zero.

In this activity you build a **Learning Plan agent** that writes a personalised study plan, one module at a time. Its state is **saved after every step** to a file on your computer. You'll interrupt it halfway through on purpose, "restart" the program, and watch it **recover and continue exactly where it stopped**, without repeating or losing any work.

It also keeps two kinds of memory separate:

| | Stored in | Example |
|---|---|---|
| **User state** | `user_profiles.json` | Amara is a beginner who studies 30 minutes a day |
| **Conversation state** | `checkpoints.db` | Amara's SQL plan: 2 of 4 modules written, next step: write module 3 |

## Learning objectives

By the end of this activity, you should be able to:

1. Explain what agent state is, and the difference between user state and conversation state.
2. Describe an agent's lifecycle, and why persistence matters.
3. Explain what checkpoints are and what they store.
4. Add persistent checkpointing to a LangGraph agent with SQLite.
5. Recover a saved state after an interruption and continue the task.
6. Keep multiple users and conversations separate.

**Time:** about 90 minutes
**You'll need:** a Gemini API key, and the Lesson 8 LangGraph activity completed.

> **How to use this notebook:** run the cells from top to bottom. Read each explanation before running the code below it. Cells marked **Try it** invite you to change something and run again.

## 1. What is agent state?

An agent's **state** is everything it needs to know to carry on with its work: what it's doing, what it has done so far, and what comes next.

Think of a video game. Your **save file** stores your level, your inventory and where you're standing. Load it tomorrow and you continue from the same spot. Agent state works the same way.

For our Learning Plan agent, the state includes:

| Field | Example |
|---|---|
| Who it's working for | `user_id: "amara"` |
| The task | `goal: "Learn SQL for data analysis"` |
| The plan | `modules: ["SQL basics", "Filtering", "Joins", "Mini project"]` |
| Progress | `current_module: 2` |
| Work done so far | `lessons: [module 1, module 2]` |
| Where it is in the process | `status: "writing"` |
| The conversation | `messages: [...]` |

In Lessons 8 and 9 you used a `TypedDict` to define state. That's **structured state**: every field has a name and a type, so you always know exactly what's stored.

## 2. User state vs conversation state

Not all state belongs to the same place. Ask: **how long should this be remembered, and who does it belong to?**

| | User state | Conversation state |
|---|---|---|
| **Belongs to** | A person | One conversation or task (a **thread**) |
| **Lasts** | Across all their conversations | For that thread only |
| **Example** | "Amara is a beginner, 30 min/day, has finished 1 plan" | "This SQL plan: 2 of 4 modules written" |
| **In Lesson 6** | `memory.json` | the `history` list |
| **In this activity** | `user_profiles.json` | `checkpoints.db`, managed by LangGraph |

One user can have **many threads**. Amara might have one thread for a SQL plan and another for an Excel plan. Both threads share her **user state** (her level, her study time), but each has its own **conversation state**.

```text
  USER STATE (user_profiles.json)           CONVERSATION STATE (checkpoints.db)
  ┌──────────────────────────┐              ┌───────────────────────────────┐
  │ amara: beginner, 30 min  │ ───────────► │ thread amara-1: SQL plan      │
  │        completed: 1 plan │ ───────────► │ thread amara-2: Excel plan    │
  ├──────────────────────────┤              ├───────────────────────────────┤
  │ ben: intermediate, 60 min│ ───────────► │ thread ben-1: website plan    │
  └──────────────────────────┘              └───────────────────────────────┘
```

At the start of each task, the agent **copies** the user's profile into the thread's state. When the task finishes, it **updates** the user's profile ("completed 1 more plan").

## 3. The agent lifecycle

Every task goes through a lifecycle:

```text
 CREATED ──► RUNNING ──────────────────────────► COMPLETED ──► (follow-up questions)
               │  step → checkpoint                ▲
               │  step → checkpoint                │
               │  step → checkpoint                │
               ▼                                   │
          INTERRUPTED ──► (state is safe on disk) ─► RESUMED
          (crash, restart,
           closed laptop,
           waiting for a human)
```

| Stage | What happens to the state |
|---|---|
| **Created** | A new thread starts with the user's request |
| **Running** | Each step reads the state, does its work, and returns updates. A **checkpoint** is saved after every step. |
| **Interrupted** | The program stops unexpectedly. Nothing new is saved, but every finished step already is. |
| **Resumed** | The latest checkpoint is loaded, and the task continues from the next unfinished step |
| **Completed** | The final state is saved. The thread can still receive follow-up questions. |

## 4. Why persistence matters

**Persistence** means saving state somewhere that survives the program stopping, like a file or a database.

In Lesson 8 you used `InMemorySaver`. It keeps checkpoints in the computer's memory, so they vanish when the program stops.

| | `InMemorySaver` (Lesson 8) | `SqliteSaver` (this activity) |
|---|---|---|
| Where checkpoints live | The program's memory | A file on disk: `checkpoints.db` |
| Survives a crash or restart | No | Yes |
| Good for | Learning, quick tests | Real apps with one computer |

Real apps need persistence because:

- **Long tasks fail halfway.** Without it, a crash at step 9 of 10 means starting again, and paying for all those LLM calls twice.
- **Servers restart.** Apps are updated and restarted all the time.
- **Humans are slow.** A human-in-the-loop approval (Lesson 8) might take hours, and the program can't wait in memory that long.
- **Many users at once.** Each user's conversations must be stored separately and safely.
- **Debugging.** A saved history shows exactly what the agent did, step by step.

Production systems often use bigger databases, such as Postgres. They work the same way: you only change the checkpointer.

## 5. Checkpoints

A **checkpoint** is a snapshot of the state, saved automatically after every step of the graph. Each checkpoint stores:

| Part | Meaning |
|---|---|
| **values** | The whole state at that moment |
| **next** | Which step should run next (empty when the task is finished) |
| **step number** | How far through the run it was |
| **checkpoint id** and **parent id** | So checkpoints form a chain, from oldest to newest |

All the checkpoints for one conversation share a **thread id**:

```text
thread "amara-1":

 checkpoint 0 ─► checkpoint 1 ─► checkpoint 2 ─► checkpoint 3 ─► checkpoint 4 ...
 (request)      (profile        (outline        (module 1       (module 2
                 loaded)         planned)        written)        written)
                                                                 next: write_module
```

If the program stops after checkpoint 4, LangGraph knows two things: the state (2 modules written) and the **next step** (write module 3). That's everything it needs to continue.

## 6. Saving and recovering state

LangGraph handles saving for you. You only need a few commands:

| Goal | Code |
|---|---|
| Turn on saving | `graph.compile(checkpointer=checkpointer)` |
| Run a task on a thread (saves after every step) | `app.invoke(input, {"configurable": {"thread_id": "amara-1"}})` |
| Look at the latest saved state | `app.get_state(config)` gives `.values` and `.next` |
| See every saved checkpoint | `app.get_state_history(config)` |
| **Continue after an interruption** | `app.invoke(None, config)` |
| Delete a thread | `checkpointer.delete_thread("amara-1")` |

The key idea is `app.invoke(None, config)`. Passing `None` instead of new input means: *"don't start anything new; load the latest checkpoint for this thread and carry on from its next step."*

The step that was interrupted runs **again from the beginning**, because it never finished, so it was never saved. Steps that already finished are **not** repeated.

### Setup

We need one extra package this time: `langgraph-checkpoint-sqlite`, which saves checkpoints to a SQLite file. SQLite is a small database that lives in a single file and comes built into Python.

In [ ]:
!pip -q install langchain langgraph langchain-google-genai langgraph-checkpoint-sqlite

In [ ]:
import os
import getpass

os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter your Gemini API key: ")

print("API key loaded.")

## 7. Build a simple agent with structured state

### User state: profiles in a JSON file

First, the **user state**: one JSON file with an entry per user, just like `memory.json` in Lesson 6. New users get a default profile.

In [ ]:
import json
import operator
from typing import Annotated, TypedDict

PROFILES_FILE = "user_profiles.json"
CHECKPOINT_DB = "checkpoints.db"
NUM_MODULES = 4

# Start fresh every time you run the notebook from the top.
for path in (PROFILES_FILE, CHECKPOINT_DB):
    if os.path.exists(path):
        os.remove(path)


DEFAULT_PROFILE = {"level": "beginner", "minutes_per_day": 30, "completed_plans": []}


def load_profiles() -> dict:
    # Read every user profile. Returns an empty dict if the file doesn't exist yet.
    if not os.path.exists(PROFILES_FILE):
        return {}
    with open(PROFILES_FILE, "r", encoding="utf-8") as f:
        return json.load(f)


def get_profile(user_id: str) -> dict:
    # Return one user's profile, or a default profile for a new user.
    return load_profiles().get(user_id, {"name": user_id.title(), **DEFAULT_PROFILE})


def save_profile(user_id: str, profile: dict) -> None:
    # Save one user's profile, keeping everyone else's.
    profiles = load_profiles()
    profiles[user_id] = profile
    with open(PROFILES_FILE, "w", encoding="utf-8") as f:
        json.dump(profiles, f, ensure_ascii=False, indent=2)


save_profile("amara", {"name": "Amara", "level": "beginner", "minutes_per_day": 30, "completed_plans": []})
save_profile("ben", {"name": "Ben", "level": "intermediate", "minutes_per_day": 60, "completed_plans": []})

print(open(PROFILES_FILE).read())

### Conversation state: `PlanState`

Now the **structured state** for one task. Two fields have reducers:

- `messages` uses `add_messages` (Lesson 8), so new messages are added to the conversation.
- `lessons` uses `operator.add` (Lesson 9), so each written module is **added** to the list.

Everything else is replaced when a step updates it.

In [ ]:
from langgraph.graph.message import add_messages


class PlanState(TypedDict):
    messages: Annotated[list, add_messages]          # the conversation in this thread
    user_id: str                                     # who this thread belongs to
    profile: dict                                    # a copy of the user's profile, loaded at the start
    goal: str                                        # what the user wants to learn
    modules: list[str]                               # the plan's module titles
    current_module: int                              # how many modules are written so far
    lessons: Annotated[list[dict], operator.add]     # written modules: {"title": ..., "content": ...}
    status: str                                      # "planning", "writing" or "done"
    final_plan: str                                  # the finished plan


print("PlanState fields:", list(PlanState.__annotations__))

### The model, prompts and steps

The agent has five steps (nodes):

| Node | What it does | Reads | Writes |
|---|---|---|---|
| `load_profile` | Copies the user's profile into the thread | `user_id` | `profile`, `status` |
| `plan_outline` | Plans the module titles (structured output) | `goal`, `profile` | `modules`, `current_module`, `status` |
| `write_module` | Writes **one** module, then loops | `modules`, `current_module` | `lessons`, `current_module` |
| `finalize` | Assembles the plan and updates the user's profile | `lessons` | `final_plan`, `status`, `messages` |
| `chat` | Answers follow-up questions about a finished plan | `final_plan`, `messages` | `messages` |

Writing modules **one per step** is the important design choice. Each finished module gets its own checkpoint, so an interruption only loses the module being written at that moment.

`write_module` also contains our **interruption simulator**: if `CRASH_AT_MODULE` matches the module it's about to write, it raises an error, as if the program had suddenly stopped.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

MODEL = "gemini-3.8-flash"
llm = ChatGoogleGenerativeAI(model=MODEL)

CRASH_AT_MODULE = None   # set to a module number to simulate an interruption


class SimulatedCrash(Exception):
    # Raised on purpose to simulate the program being interrupted.
    pass


class Outline(BaseModel):
    """The plan's structure."""
    modules: list[str] = Field(description=f"Exactly {NUM_MODULES} short module titles, from first to last")


OUTLINE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "You design short, practical learning plans. The learner is at {level} level and can study "
               "{minutes} minutes per day. Create exactly {num} modules that build on each other."),
    ("human", "Learning goal: {goal}"),
])

MODULE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "You write one module of a learning plan for a {level} learner who studies {minutes} minutes "
               "per day. In under 90 words, give: what to learn, one hands-on practice task, and how many "
               "days it should take. Plain text, no headings."),
    ("human", "Learning goal: {goal}\nFull plan: {modules}\nWrite module {number}: {title}"),
])

CHAT_PROMPT = ("You are a friendly study coach. You already wrote this learning plan for {name}:\n\n{plan}\n\n"
               "Answer their questions about it briefly and encouragingly.")


print("Model and prompts ready.")

In [ ]:
def load_profile(state: PlanState) -> dict:
    # Copy the user's long-term profile into this thread's state.
    profile = get_profile(state["user_id"])
    print(f"  [load_profile] {profile['name']}: {profile['level']}, {profile['minutes_per_day']} min/day, "
          f"{len(profile['completed_plans'])} plan(s) completed before")
    return {"profile": profile, "status": "planning"}


def plan_outline(state: PlanState) -> dict:
    # Decide the plan's modules.
    chain = OUTLINE_PROMPT | llm.with_structured_output(Outline)
    outline = chain.invoke({"goal": state["goal"], "level": state["profile"]["level"],
                            "minutes": state["profile"]["minutes_per_day"], "num": NUM_MODULES})
    modules = outline.modules[:NUM_MODULES]
    print(f"  [plan_outline] {modules}")
    return {"modules": modules, "current_module": 0, "status": "writing"}


def write_module(state: PlanState) -> dict:
    # Write the next module. This is the slow, multi-step part of the task.
    number = state["current_module"] + 1
    title = state["modules"][state["current_module"]]

    if CRASH_AT_MODULE == number:
        raise SimulatedCrash(f"Interrupted while writing module {number}!")

    chain = MODULE_PROMPT | llm
    reply = chain.invoke({"goal": state["goal"], "level": state["profile"]["level"],
                          "minutes": state["profile"]["minutes_per_day"],
                          "modules": ", ".join(state["modules"]), "number": number, "title": title})
    print(f"  [write_module] module {number}/{len(state['modules'])} written: {title}")
    return {"lessons": [{"title": title, "content": reply.text}], "current_module": number}


def finalize(state: PlanState) -> dict:
    # Assemble the plan, and record it in the user's long-term profile.
    parts = [f"Learning plan: {state['goal']}\n"]
    for i, lesson in enumerate(state["lessons"], 1):
        parts.append(f"Module {i}: {lesson['title']}\n{lesson['content']}\n")
    final_plan = "\n".join(parts)

    profile = get_profile(state["user_id"])
    profile["completed_plans"] = profile["completed_plans"] + [state["goal"]]
    save_profile(state["user_id"], profile)
    print(f"  [finalize] plan saved; {profile['name']} has now completed {len(profile['completed_plans'])} plan(s)")

    return {"final_plan": final_plan, "status": "done",
            "messages": [AIMessage(content=f"Your {len(state['lessons'])}-module plan is ready!")]}


def chat(state: PlanState) -> dict:
    # Answer follow-up questions about a finished plan, using the saved conversation.
    system = CHAT_PROMPT.format(name=state["profile"]["name"], plan=state["final_plan"])
    history = [m for m in state["messages"] if isinstance(m, (HumanMessage, AIMessage))]
    reply = llm.invoke([SystemMessage(content=system)] + history)
    return {"messages": [AIMessage(content=reply.text)]}


def route_start(state: PlanState) -> str:
    # A finished plan means this is a follow-up question; otherwise start (or restart) the task.
    return "chat" if state.get("status") == "done" else "load_profile"


def route_modules(state: PlanState) -> str:
    # Keep writing modules until every module in the plan is done.
    return "write_module" if state["current_module"] < len(state["modules"]) else "finalize"


print("Nodes and routing ready.")

`route_start` decides where each run begins: a thread whose plan is already `done` goes to `chat` (it's a follow-up question); anything else starts the task. `route_modules` keeps looping back to `write_module` until every module is written.

## 8. Add checkpointing with LangGraph

Now we give the agent persistence. `SqliteSaver` saves every checkpoint to `checkpoints.db`.

`check_same_thread=False` is a technical setting that lets LangGraph use the database connection safely. You'll always include it.

In [ ]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import StateGraph, START, END


def get_checkpointer(db_path: str = CHECKPOINT_DB) -> SqliteSaver:
    # A checkpointer that saves every checkpoint to a SQLite file on disk.
    connection = sqlite3.connect(db_path, check_same_thread=False)
    return SqliteSaver(connection)


def build_graph(checkpointer):
    graph = StateGraph(PlanState)

    graph.add_node("load_profile", load_profile)
    graph.add_node("plan_outline", plan_outline)
    graph.add_node("write_module", write_module)
    graph.add_node("finalize", finalize)
    graph.add_node("chat", chat)

    graph.add_conditional_edges(START, route_start, ["load_profile", "chat"])
    graph.add_edge("load_profile", "plan_outline")
    graph.add_edge("plan_outline", "write_module")
    graph.add_conditional_edges("write_module", route_modules, ["write_module", "finalize"])
    graph.add_edge("finalize", END)
    graph.add_edge("chat", END)

    return graph.compile(checkpointer=checkpointer)


app = build_graph(get_checkpointer())
print(app.get_graph().draw_mermaid())

Paste the diagram into [mermaid.live](https://mermaid.live). Notice the loop on `write_module`.

The only difference from an agent without persistence is one argument: `checkpointer=checkpointer` in `compile()`. Everything else is the same graph you'd build in Lesson 8.

A few small helpers make the next sections easier to read:

- `start_task` runs a new task on a thread, and catches the simulated crash.
- `show_status` prints what's saved for a thread, including its checkpoint history.
- `list_threads` looks directly inside the SQLite file.

In [ ]:
def thread_config(thread_id: str) -> dict:
    return {"configurable": {"thread_id": thread_id}}


def start_task(app, user_id: str, goal: str, thread_id: str):
    # Start a new learning-plan task on a thread. Returns the final state, or None if interrupted.
    print(f"\nStarting task on thread '{thread_id}' for user '{user_id}': {goal}")
    first_state = {"messages": [HumanMessage(content=f"Make me a learning plan: {goal}")],
                   "user_id": user_id, "goal": goal, "lessons": []}
    try:
        return app.invoke(first_state, thread_config(thread_id))
    except SimulatedCrash as crash:
        print(f"\n  INTERRUPTED: {crash}")
        print(f"  Progress up to the last finished step is saved. Resume thread '{thread_id}' to continue.")
        return None


def show_status(app, thread_id: str) -> None:
    # Print what is saved for a thread, and its checkpoint history.
    config = thread_config(thread_id)
    saved = app.get_state(config)
    if not saved.values:
        print(f"No saved state for thread '{thread_id}'.")
        return
    v = saved.values
    print(f"\nThread '{thread_id}'")
    print(f"  user:            {v.get('user_id')}")
    print(f"  goal:            {v.get('goal')}")
    print(f"  status:          {v.get('status')}")
    print(f"  modules written: {v.get('current_module', 0)} / {len(v.get('modules', []))}")
    print(f"  messages:        {len(v.get('messages', []))}")
    print(f"  next step:       {saved.next[0] if saved.next else '(finished)'}")
    history = list(app.get_state_history(config))
    print(f"  checkpoints:     {len(history)} saved (newest first):")
    for snapshot in history[:8]:
        nxt = snapshot.next[0] if snapshot.next else "-"
        print(f"    step {snapshot.metadata.get('step'):>3}  next: {nxt:13} modules done: "
              f"{snapshot.values.get('current_module', 0)}")


def list_threads(db_path: str = CHECKPOINT_DB) -> list[tuple]:
    # List every thread saved in the SQLite file, with its number of checkpoints.
    if not os.path.exists(db_path):
        return []
    with sqlite3.connect(db_path) as conn:
        try:
            return conn.execute(
                "SELECT thread_id, COUNT(*) FROM checkpoints GROUP BY thread_id ORDER BY thread_id").fetchall()
        except sqlite3.OperationalError:   # the file exists but nothing has been saved yet
            return []


print("Helpers ready.")

## 9. Run the agent and save its state

Let's run a complete task for **Ben**, on thread `ben-1`. Watch each step print as it runs. Every step is saved to `checkpoints.db` as it finishes.

In [ ]:
result = start_task(app, "ben", "Build a personal website with HTML and CSS", "ben-1")

print("\n" + result["final_plan"])

### Look at what was saved

The task is finished, but everything about it is still in `checkpoints.db`. `show_status` reads it back.

In [ ]:
show_status(app, "ben-1")

Read the checkpoint history from the bottom up: the profile loaded, the outline planned, the modules written one by one, then the plan finalised. `next` shows the step each checkpoint was waiting to run.

### Open the files yourself

Both files are ordinary files on your computer. In Colab, you can see them in the **Files** panel on the left.

In [ ]:
# User state: plain JSON you can read.
print(open(PROFILES_FILE).read())

# Conversation state: rows in a SQLite table.
with sqlite3.connect(CHECKPOINT_DB) as conn:
    rows = conn.execute("SELECT thread_id, checkpoint_id, parent_checkpoint_id FROM checkpoints").fetchall()

print(f"{len(rows)} checkpoint rows in {CHECKPOINT_DB}. The first three:")
for row in rows[:3]:
    print("  thread:", row[0], "| checkpoint:", row[1][:13], "... | parent:", (row[2] or "none")[:13])

Ben's profile now records 1 completed plan. That's the **user state** being updated.

The SQLite rows show the chain of checkpoints: each one points to its **parent**. The state itself is stored in a compact format, which is why we read it with `get_state` rather than opening the database directly.

## 10. Simulate an interruption

Now **Amara** asks for a plan. This time, we interrupt the agent while it's writing **module 3**, as if the program had suddenly stopped.

In [ ]:
CRASH_AT_MODULE = 3

result = start_task(app, "amara", "Learn SQL for data analysis", "amara-1")
print("\nResult returned:", result)

The task stopped, and `start_task` returned `None`, so there's no finished plan. But was the work lost? Let's look at what's saved for thread `amara-1`.

In [ ]:
show_status(app, "amara-1")

Everything up to the interruption is safe:

- **2 of 4 modules** are written and saved.
- **next step: `write_module`**. LangGraph knows exactly where to continue.

Module 3 isn't saved, because that step never finished.

## 11. Recover the previous state and continue the task

To prove the state really comes from the **file**, not from Python's memory, we simulate a **full restart**:

1. Close the database connection and throw away the `app`.
2. Build a brand-new app with a brand-new connection to `checkpoints.db`.
3. Resume thread `amara-1` with `app.invoke(None, config)`.

We also switch off the crash simulator, since the "problem" has been fixed.

In [ ]:
def resume_task(app, thread_id: str):
    # Recover a thread's saved state and continue from where it stopped.
    config = thread_config(thread_id)
    saved = app.get_state(config)

    if not saved.values:
        print(f"No saved state for thread '{thread_id}'.")
        return None
    if not saved.next:
        print(f"Thread '{thread_id}' has nothing left to do (status: {saved.values.get('status')}).")
        return saved.values

    print(f"\nRecovered thread '{thread_id}': {saved.values.get('current_module', 0)} of "
          f"{len(saved.values.get('modules', []))} modules done. Next step: {saved.next[0]}")
    try:
        return app.invoke(None, config)   # None = "no new input, just continue from the checkpoint"
    except SimulatedCrash as crash:
        print(f"\n  INTERRUPTED again: {crash}")
        return None


# 1. "Restart the program": close the connection and throw the app away.
CRASH_AT_MODULE = None
app.checkpointer.conn.close()
del app

# 2. A brand-new app, reading from the same file on disk.
app = build_graph(get_checkpointer())

# 3. Recover and continue.
result = resume_task(app, "amara-1")
print("\n" + result["final_plan"])

Look at the `[write_module]` lines: the agent continued at **module 3**. It didn't re-plan, re-load the profile, or rewrite modules 1 and 2. Let's confirm that nothing was repeated or lost.

In [ ]:
titles = [lesson["title"] for lesson in result["lessons"]]

print("Modules in the final plan:", titles)
print("Exactly", NUM_MODULES, "modules:", len(titles) == NUM_MODULES)
print("No duplicates:", len(titles) == len(set(titles)))

show_status(app, "amara-1")

> **Try it:** start a new thread with `CRASH_AT_MODULE = 2`, then resume it. Then try `CRASH_AT_MODULE = 1`. Does the agent always continue from the right place?

## 12. Test multiple conversations and users

A persistent agent usually serves many people at once. Let's check that their state stays separate, and that user state is shared across a user's own threads.

### Follow-up questions in a saved conversation

Amara comes back later with a question about her plan. Because thread `amara-1` is saved, the agent has the whole plan and conversation. `route_start` sees the plan is `done` and sends the question to `chat`.

In [ ]:
def ask(app, thread_id: str, question: str) -> str:
    # Ask a follow-up question in an existing thread.
    result = app.invoke({"messages": [HumanMessage(content=question)]}, thread_config(thread_id))
    return result["messages"][-1].text


print("Coach (amara-1):", ask(app, "amara-1", "I have a busy week. Which module should I start with, and why?"))

### A second thread for the same user

Amara starts a **new** plan on thread `amara-2`. Watch the `[load_profile]` line: her **user state** carries over (she's now completed 1 plan), but the new thread's **conversation state** starts fresh.

In [ ]:
result_2 = start_task(app, "amara", "Learn Excel for budgeting", "amara-2")
print("\nModules in amara-2:", result_2["modules"])

### Check that everything is separate

In [ ]:
amara_1 = app.get_state(thread_config("amara-1")).values
amara_2 = app.get_state(thread_config("amara-2")).values
ben_1 = app.get_state(thread_config("ben-1")).values
profiles = load_profiles()

checks = [
    ("Three threads are saved in the SQLite file", {t for t, _ in list_threads()} == {"amara-1", "amara-2", "ben-1"}),
    ("Each thread kept its own goal", len({amara_1["goal"], amara_2["goal"], ben_1["goal"]}) == 3),
    ("Ben's thread used Ben's profile (intermediate)", ben_1["profile"]["level"] == "intermediate"),
    ("Amara's threads used Amara's profile (beginner)", amara_2["profile"]["level"] == "beginner"),
    ("Amara's follow-up stayed in amara-1", len(amara_1["messages"]) > len(amara_2["messages"])),
    ("Amara's user state counts 2 completed plans", len(profiles["amara"]["completed_plans"]) == 2),
    ("Ben's user state counts 1 completed plan", len(profiles["ben"]["completed_plans"]) == 1),
    ("The recovered plan has exactly 4 modules", len(amara_1["lessons"]) == NUM_MODULES),
]

for name, ok in checks:
    print(("PASS " if ok else "CHECK"), name)
print(f"\n{sum(ok for _, ok in checks)}/{len(checks)} checks passed.")

print("\nThreads in the database:", list_threads())

### What to look for

- **Isolation:** each thread has its own goal, modules and conversation. Nothing leaks between threads.
- **Shared user state:** Amara's level and study time apply to both of her threads, and her completed-plan count went up after each one.
- **Durability:** the agent recovered `amara-1` from the file after a "restart", and nothing was repeated or lost.

> **Try it:** ask a follow-up in `ben-1`, for example "How long will the whole plan take me?" Does the answer use Ben's 60 minutes a day?

## 13. Challenge

### Part A: An approval that survives a restart

In Lesson 8, the human approved a message while the program kept running. With persistence, a human can approve **hours later**, even after the program has restarted.

1. Add a `review_outline` node between `plan_outline` and `write_module`. It calls `interrupt()` with the module titles and asks "Approve this outline? (yes/no)". (Import it with `from langgraph.types import interrupt, Command`.)
2. If the answer is `"no"`, set `modules` to a shorter list of your choice, or send the run back to `plan_outline`.
3. Run a task. It will pause at the review.
4. Simulate a restart: close the connection and rebuild the app, as in Section 11.
5. Resume with `app.invoke(Command(resume="yes"), config)`. The plan should continue from the review.

### Part B: Forget a user

Privacy laws often give people the right to have their data deleted. Write `forget_user(app, user_id)` that:

- deletes every thread belonging to that user, using `app.checkpointer.delete_thread(thread_id)` (threads starting with `user_id + "-"`),
- removes their entry from `user_profiles.json`.

Test it on Ben, then check `list_threads()` and the JSON file.

### Stretch: time travel

`app.get_state_history(config)` returns every checkpoint. Pick the checkpoint from just after module 2 was written, then resume **from that point** by passing its config (`snapshot.config`) to `app.invoke(None, snapshot.config)`. The agent rewrites modules 3 and 4 as a new branch of the history. When would this be useful?

In [ ]:
# Part A: your review_outline node.

from langgraph.types import interrupt, Command


def review_outline(state: PlanState) -> dict:
    # TODO: call interrupt() with the module titles and a question, e.g.
    #       decision = interrupt({"question": "Approve this outline? (yes/no)", "modules": state["modules"]})
    #       If the decision isn't "yes", return a different "modules" list. Otherwise return {}.
    raise NotImplementedError("TODO: complete review_outline")


# Then: add the node to build_graph() between plan_outline and write_module, rebuild the app, and test:
# app = build_graph(get_checkpointer())
# start_task(app, "amara", "Learn Python basics", "amara-3")     # pauses at review_outline
# app.checkpointer.conn.close(); app = build_graph(get_checkpointer())   # "restart"
# result = app.invoke(Command(resume="yes"), thread_config("amara-3"))

In [ ]:
# Part B: forget a user.

def forget_user(app, user_id: str) -> None:
    # TODO 1: for every thread id from list_threads() that starts with user_id + "-",
    #         call app.checkpointer.delete_thread(thread_id)
    # TODO 2: load the profiles, remove profiles[user_id] if it exists, and write the file back
    raise NotImplementedError("TODO: complete forget_user")


# forget_user(app, "ben")
# print(list_threads())
# print(open(PROFILES_FILE).read())

## Recap

| Concept | What you built |
|---|---|
| **Structured state** | `PlanState`: every field named and typed, with reducers where values are added |
| **User state** | `user_profiles.json`, shared across a user's threads and updated when a plan finishes |
| **Conversation state** | One thread per task, saved in `checkpoints.db` |
| **Lifecycle** | Created → running → interrupted → resumed → completed → follow-up |
| **Checkpoints** | Saved automatically after every step: values, next step, and history |
| **Persistence** | `SqliteSaver` instead of `InMemorySaver`: one argument changes everything |
| **Recovery** | `app.invoke(None, config)` continues from the latest checkpoint, without repeating finished steps |
| **Isolation** | Each thread is separate; each user's profile is separate |

**Next:** complete the starter file `lesson_10_persistent_agent.py`, run its tests, and submit your work. See `README.md` for the submission checklist.